# Lecture 37: Classifier Accuracy and Updating Probabilities
v.ekc

Two halves today: wrapping up k-NN accuracy, then a new (and final!) big idea — **updating probabilities with evidence**. A test comes back positive: what's the chance you actually have the disease? The answer surprises almost everyone. (Slides: KL Lecture 37 deck.)

**Today's sections:**
1. The k-NN functions
2. Accuracy recap
3. Updating probabilities
4. Decisions — the medical test
5. Subjective probabilities

In [ ]:
from datascience import *
import numpy as np
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. The k-NN Functions

The full pipeline from last time — run these setup cells and move on.

In [ ]:
def distance(pt1, pt2):
    """Return the distance between two points, represented as arrays"""
    return np.sqrt(sum((pt1-pt2)**2))

In [ ]:
def row_distance(row1, row2):
    """Return the distance between two numerical rows of a table"""
    row1_array = np.array(row1)
    row2_array = np.array(row2)
    return distance(row1_array,row2_array)

In [ ]:
def distances(training, example):
    """
    Compute distance between example and every row in training.
    Return training augmented with Distance column
    """
    distances = make_array()
    attributes_only = training.drop('Class')
    
    for row in attributes_only.rows:
        distances = np.append(distances,row_distance(row,example)) # append distance between row and example
        
    return training.with_column('Distance_to_ex', distances)

In [ ]:
def closest(training, example, k):
    """
    Return a table of the k closest neighbors to example
    """
    sorted_distances = distances(training,example).sort('Distance_to_ex')
    return sorted_distances.take(np.arange(k))

In [ ]:
def majority_class(topk):
    """
    Return the class with the highest count
    """
    return topk.group('Class').sort('count', descending=True).column(0).item(0)

In [ ]:
def classify(training, example, k):
    """
    Return the majority class among the 
    k nearest neighbors of example
    """
    return majority_class(closest(training, example, k))

---
## 2. Accuracy Recap

Breast-cancer data: shuffle, split, test — then the CKD data with and without standard units.

> ⚠️ *Note: the original notebook had a typo (`eshuffled`) that made the standardized comparison silently reuse the wrong table — fixed here.*

In [ ]:
# Data
patients = Table.read_table('breast-cancer.csv').drop('ID')
attributes = patients.drop('Class')
attributes.show(3)

In [ ]:
patients.num_rows

In [ ]:
shuffled = patients.sample(with_replacement=False) # Randomly permute the rows
training_set = shuffled.take(np.arange(342))
test_set  = shuffled.take(np.arange(342, 683))

In [ ]:
def evaluate_accuracy(training, test, k):
    """Return the proportion of correctly classified examples 
    in the test set"""
    test_attributes = test.drop('Class')
    num_correct = 0
    for i in np.arange(test.num_rows):
        c = classify(training, test_attributes.row(i), k)
        is_correct = (c == test.column('Class').item(i))
        num_correct = num_correct + is_correct
    return num_correct / test.num_rows

In [ ]:
evaluate_accuracy(training_set, test_set, 5)

In [ ]:
evaluate_accuracy(training_set, test_set, 3)

In [ ]:
evaluate_accuracy(training_set, test_set, 11)

In [ ]:
evaluate_accuracy(training_set, test_set, 1)

# Standardize if Necessary

In [ ]:
def standard_units(x):
    return (x - np.average(x)) / np.std(x)

In [ ]:
ckd = Table.read_table('ckd.csv')
ckd = ckd.relabeled('Blood Glucose Random', 'Glucose').select('Glucose', 'Hemoglobin', 'White Blood Cell Count', 'Class')
ckd.show(5)

In [ ]:
ckd_new = ckd.select('Class').with_columns(
    'Glucose_su', standard_units(ckd.column('Glucose')),
    'Hemoglobin_su', standard_units(ckd.column('Hemoglobin')),
    'WBC_su', standard_units(ckd.column('White Blood Cell Count'))
)

In [ ]:
ckd_new

In [ ]:
shuffled = ckd_new.sample(with_replacement=False) 
training_set = shuffled.take(np.arange(74))
test_set  = shuffled.take(np.arange(74, 148))

In [ ]:
evaluate_accuracy(training_set, test_set, 3)

In [ ]:
shuffled = ckd.sample(with_replacement=False) 
training_set = shuffled.take(np.arange(74))
test_set  = shuffled.take(np.arange(74, 148))

In [ ]:
evaluate_accuracy(training_set, test_set, 3)

---
## 3. Updating Probabilities 🌳

New question type: I pick a student at random; they tell me they've **declared** a major. Second year or third year? Evidence should *update* our answer. (KL deck: Lecture 37.)

### 📋 Board Reference — updating a probability

| Piece | Meaning |
|---|---|
| **prior** | P(class) before seeing evidence |
| **likelihood** | P(evidence \| class) |
| tree branch | prior × likelihood |
| **posterior** | branch ÷ (sum of all branches with that evidence) |
| Bayes' rule | P(class \| evidence) = P(class & evidence) / P(evidence) |

In [ ]:
n = 100
second = round(n * 0.6)
third = round(n * 0.4)

year = np.array(['Second'] * second + ['Third'] * third)
major = np.array(['Declared'] * (round(second * 0.5)) + ['Undeclared'] * (round(second * 0.5)) + \
                 ['Declared'] * (round(third * 0.8))  + ['Undeclared'] * (round(third * 0.2)))
                 
students = Table().with_columns(
    'Year', year,
    'Major', major
)

In [ ]:
students.show(3)

In [ ]:
students.pivot('Major', 'Year')

In [ ]:
# Verify: 60% of students are Second years, 40% are Third years
60 / (60 + 40)

In [ ]:
# Verify: 50% of Second years have Declared
30 / 60

In [ ]:
# Verify: 80% of Third years have Declared
32 / 40

In [ ]:
# Chance of second year, given that they have declared
# P(second year | declared)

30 / 62

In [ ]:
# P(third year | declared)

32 / 62

## Tree Diagram Calculation

In [ ]:
# P(second year | declared), from tree diagram

(0.6 * 0.5) / (0.6 * 0.5 + 0.4 * 0.8)

### Try It 1: Flip the evidence 😊

1. What's P(second year | **un**declared)? Compute it twice: once by counting from the pivot table, once from the tree diagram (prior × likelihood over branches).

In [ ]:
# 1. P(second year | undeclared) — count, then tree


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Undeclared students: 30 second-years, 8 third-years.*

```python
# 1. counting
30 / (30 + 8)                                # ≈ 0.789

# 1. tree diagram
(0.6 * 0.5) / (0.6 * 0.5 + 0.4 * 0.2)        # same!
```

</details>

---
## 4. Decisions — the Medical Test 🏥

A disease hits 1 in 1000 people. The test is excellent: 99% right on the sick, 95% right on the healthy. You test positive. What's the chance you're sick? **Guess first.**

In [ ]:
def create_population(prior_disease_prob, n):
    disease = round(n * prior_disease_prob)
    no_disease = round(n * (1 - prior_disease_prob))

    status = np.array(['Disease'] * disease  +  ['No disease'] * no_disease)
    result = np.array(['Test +'] * (round(disease * 0.99)) + ['Test -'] * (round(disease * 0.01)) +\
                      ['Test +'] * (round(no_disease * 0.05)) + ['Test -'] * (round(no_disease * 0.95)) )
                 
    t = Table().with_columns(
    'Status', status,
    'Test Result', result
    )
    return t.pivot('Test Result', 'Status')

In [ ]:
create_population(1/1000, 100000)

In [ ]:
99 / (99 + 4995)

In [ ]:
#P(disease | tested +)

# = P(disease & tested +) / P(tested +)

(0.001 * 0.99) / (0.001*0.99 + 0.999*0.05)

> **About 2%.** Not 99% — the disease is so rare that false positives from the huge healthy population swamp the true positives. This is the **base rate fallacy**, and even doctors get it wrong. The population count (`create_population`) and Bayes' rule agree exactly.

---
## 5. Subjective Probabilities

The answer *depends on the prior*. A patient with symptoms isn't a random member of the population — watch the posterior move as the prior rises:

In [ ]:
# P(disease | tested +)

# = P(disease & tested +) / P(tested +)

# if prior probability of disease is 1/10

(0.1 * 0.99) / (0.1*0.99 + 0.9*0.05)

In [ ]:
create_population(1/10, 10000)

In [ ]:
990/(990+450)

In [ ]:
# P(disease | tested +)
# if prior probability of disease is 0.5

(0.5 * 0.99) / (0.5*0.99 + 0.5*0.05)

In [ ]:
create_population(0.5, 10000)

In [ ]:
4950/(4950+250)

---
> **Today's takeaway:** honest accuracy comes from held-out data, and evidence updates probability via prior × likelihood. Lab 10's icon arrays and cancer-screening questions are exactly today's section 4.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Bayes in one line
```python
posterior = (prior * likelihood) / total_prob_of_evidence
```